# 📊 Datathon 2026 — Phase 1: Setup & EDA Overview

**Mục tiêu**: Hiểu toàn bộ dữ liệu trước khi làm bài

### Checklist
- [x] Load tất cả 14 file CSV
- [x] Kiểm tra `shape`, `dtypes`, `head()`
- [x] Kiểm tra missing values
- [x] Kiểm tra duplicates
- [x] Xác nhận FK constraints
- [x] Thống kê phân bổ các cột quan trọng
- [x] Vẽ time series Revenue/COGS

## 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '../data/'
RANDOM_SEED = 42

print('✅ Setup complete')

## 2 — Load All 14 CSV Files

In [ ]:
# ======================== MASTER TABLES ========================
products   = pd.read_csv(DATA_DIR + 'products.csv')
customers  = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
geography  = pd.read_csv(DATA_DIR + 'geography.csv')
promotions = pd.read_csv(DATA_DIR + 'promotions.csv', parse_dates=['start_date', 'end_date'])

# ======================== TRANSACTION TABLES ========================
orders      = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
payments    = pd.read_csv(DATA_DIR + 'payments.csv')
shipments   = pd.read_csv(DATA_DIR + 'shipments.csv', parse_dates=['ship_date', 'delivery_date'])
returns     = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
reviews     = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])

# ======================== ANALYTICAL TABLES ========================
sales_train = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])
sample_sub  = pd.read_csv(DATA_DIR + 'sample_submission.csv', parse_dates=['Date'])

# ======================== OPERATIONAL TABLES ========================
inventory   = pd.read_csv(DATA_DIR + 'inventory.csv', parse_dates=['snapshot_date'])
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])

print('✅ All 14 CSV files loaded successfully!')

## 3 — Shape & Date Range Summary

In [ ]:
tables = {
    'products': products, 'customers': customers, 'geography': geography,
    'promotions': promotions, 'orders': orders, 'order_items': order_items,
    'payments': payments, 'shipments': shipments, 'returns': returns,
    'reviews': reviews, 'sales_train': sales_train, 'sample_sub': sample_sub,
    'inventory': inventory, 'web_traffic': web_traffic
}

summary_rows = []
for name, df in tables.items():
    row = {'Table': name, 'Rows': df.shape[0], 'Cols': df.shape[1],
           'Memory (MB)': round(df.memory_usage(deep=True).sum() / 1e6, 2)}
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df['Rows'] = summary_df['Rows'].apply(lambda x: f'{x:,}')
display(summary_df.style.set_caption('📋 Dataset Overview'))

In [ ]:
# Date ranges for time-related tables
print('📅 Date Ranges:')
print(f"  orders:      {orders['order_date'].min().date()} → {orders['order_date'].max().date()}")
print(f"  sales_train: {sales_train['Date'].min().date()} → {sales_train['Date'].max().date()}")
print(f"  sample_sub:  {sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()}")
print(f"  web_traffic: {web_traffic['date'].min().date()} → {web_traffic['date'].max().date()}")
print(f"  inventory:   {inventory['snapshot_date'].min().date()} → {inventory['snapshot_date'].max().date()}")
print(f"  promotions:  {promotions['start_date'].min().date()} → {promotions['end_date'].max().date()}")
print(f"  returns:     {returns['return_date'].min().date()} → {returns['return_date'].max().date()}")
print(f"  reviews:     {reviews['review_date'].min().date()} → {reviews['review_date'].max().date()}")
print(f"  customers:   {customers['signup_date'].min().date()} → {customers['signup_date'].max().date()}")

## 4 — Schema Inspection (dtypes & head)

In [ ]:
for name, df in tables.items():
    print(f'\n{"="*60}')
    print(f'📦 {name.upper()} — {df.shape[0]:,} rows × {df.shape[1]} cols')
    print(f'{"="*60}')
    print(df.dtypes.to_string())
    display(df.head(3))
    print()

## 5 — Missing Values Analysis

In [ ]:
print('🔍 Missing Values Summary')
print('=' * 70)

for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f'  ✅ {name:15s} — No missing values')
    else:
        print(f'  ⚠️  {name:15s} — Missing in {len(missing)} column(s):')
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f'       • {col}: {cnt:,} ({pct:.2f}%)')

In [ ]:
# Visual heatmap of missing values for key transaction tables
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
miss_tables = [('orders', orders), ('order_items', order_items), ('payments', payments),
               ('shipments', shipments), ('returns', returns), ('reviews', reviews)]

for ax, (name, df) in zip(axes.flat, miss_tables):
    miss_pct = df.isnull().mean() * 100
    colors = ['#2ecc71' if p == 0 else '#e74c3c' for p in miss_pct]
    ax.barh(miss_pct.index, miss_pct.values, color=colors)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Missing %')
    ax.set_xlim(0, max(miss_pct.max() * 1.3, 1))

plt.suptitle('Missing Values (%) — Transaction Tables', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6 — Duplicates Check

In [ ]:
print('🔁 Duplicates Check (full-row duplicates)')
print('=' * 50)
for name, df in tables.items():
    n_dup = df.duplicated().sum()
    status = '✅ None' if n_dup == 0 else f'⚠️  {n_dup:,} duplicates'
    print(f'  {name:15s} — {status}')

# Primary key uniqueness check
print('\n🔑 Primary Key Uniqueness Check')
print('=' * 50)
pk_checks = [
    ('products', products, 'product_id'),
    ('customers', customers, 'customer_id'),
    ('orders', orders, 'order_id'),
    ('returns', returns, 'return_id'),
    ('reviews', reviews, 'review_id'),
]
for name, df, pk in pk_checks:
    is_unique = df[pk].is_unique
    status = '✅ Unique' if is_unique else f'⚠️  NOT unique ({df[pk].duplicated().sum():,} dups)'
    print(f'  {name:15s}.{pk:15s} — {status}')

## 7 — Foreign Key Constraint Validation

In [ ]:
print('🔗 Foreign Key Validation')
print('=' * 70)

def check_fk(child_df, child_name, child_col, parent_df, parent_name, parent_col):
    child_set = set(child_df[child_col].dropna().unique())
    parent_set = set(parent_df[parent_col].dropna().unique())
    orphans = child_set - parent_set
    n_orphan = len(orphans)
    pct = n_orphan / len(child_set) * 100 if len(child_set) > 0 else 0
    if n_orphan == 0:
        print(f'  ✅ {child_name}.{child_col} → {parent_name}.{parent_col}: All matched')
    else:
        print(f'  ⚠️  {child_name}.{child_col} → {parent_name}.{parent_col}: '
              f'{n_orphan:,} orphan(s) ({pct:.2f}%)')
    return n_orphan

# orders → customers
check_fk(orders, 'orders', 'customer_id', customers, 'customers', 'customer_id')
# orders → geography
check_fk(orders, 'orders', 'zip', geography, 'geography', 'zip')
# order_items → orders
check_fk(order_items, 'order_items', 'order_id', orders, 'orders', 'order_id')
# order_items → products
check_fk(order_items, 'order_items', 'product_id', products, 'products', 'product_id')
# order_items → promotions (optional FK)
check_fk(order_items[order_items['promo_id'].notna()], 'order_items', 'promo_id', promotions, 'promotions', 'promo_id')
# payments → orders
check_fk(payments, 'payments', 'order_id', orders, 'orders', 'order_id')
# shipments → orders
check_fk(shipments, 'shipments', 'order_id', orders, 'orders', 'order_id')
# returns → orders
check_fk(returns, 'returns', 'order_id', orders, 'orders', 'order_id')
# returns → products
check_fk(returns, 'returns', 'product_id', products, 'products', 'product_id')
# reviews → orders
check_fk(reviews, 'reviews', 'order_id', orders, 'orders', 'order_id')
# reviews → products
check_fk(reviews, 'reviews', 'product_id', products, 'products', 'product_id')
# inventory → products
check_fk(inventory, 'inventory', 'product_id', products, 'products', 'product_id')

## 8 — Key Column Distributions

In [ ]:
# 8.1 Products — Category & Segment distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

products['category'].value_counts().plot.bar(ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Products by Category', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

products['segment'].value_counts().plot.bar(ax=axes[1], color='#e67e22', edgecolor='white')
axes[1].set_title('Products by Segment', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

products['size'].value_counts().plot.bar(ax=axes[2], color='#2ecc71', edgecolor='white')
axes[2].set_title('Products by Size', fontweight='bold')
axes[2].set_ylabel('Count')

plt.suptitle('Product Catalog Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 8.2 Products — Price & COGS distribution + Gross Margin by Segment
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(products['price'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (VND)')
axes[0].set_ylabel('Count')

axes[1].hist(products['cogs'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_title('COGS Distribution', fontweight='bold')
axes[1].set_xlabel('COGS (VND)')
axes[1].set_ylabel('Count')

products.boxplot(column='gross_margin', by='segment', ax=axes[2])
axes[2].set_title('Gross Margin by Segment', fontweight='bold')
axes[2].set_xlabel('Segment')
axes[2].set_ylabel('Gross Margin')
plt.suptitle('')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 8.3 Customers — Demographics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

customers['gender'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%',
                                             colors=['#3498db', '#e74c3c', '#95a5a6'])
axes[0].set_title('Gender Distribution', fontweight='bold')
axes[0].set_ylabel('')

age_order = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
age_counts = customers['age_group'].value_counts().reindex(age_order, fill_value=0)
age_counts.plot.bar(ax=axes[1], color='#9b59b6', edgecolor='white')
axes[1].set_title('Age Group Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

customers['acquisition_channel'].value_counts().plot.bar(ax=axes[2], color='#1abc9c', edgecolor='white')
axes[2].set_title('Acquisition Channel', fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Customer Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 8.4 Orders — Status & Payment method distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

orders['order_status'].value_counts().plot.bar(ax=axes[0], color='#2980b9', edgecolor='white')
axes[0].set_title('Order Status', fontweight='bold')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9)

orders['payment_method'].value_counts().plot.bar(ax=axes[1], color='#e67e22', edgecolor='white')
axes[1].set_title('Payment Method', fontweight='bold')
axes[1].set_ylabel('Count')

orders['device_type'].value_counts().plot.bar(ax=axes[2], color='#27ae60', edgecolor='white')
axes[2].set_title('Device Type', fontweight='bold')
axes[2].set_ylabel('Count')

plt.suptitle('Orders Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 8.5 Geography — Region distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

geography['region'].value_counts().plot.bar(ax=axes[0], color='#8e44ad', edgecolor='white')
axes[0].set_title('Zip Codes by Region', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

top_cities = geography.groupby('city')['zip'].nunique().sort_values(ascending=False).head(15)
top_cities.plot.barh(ax=axes[1], color='#2c3e50')
axes[1].set_title('Top 15 Cities by # Zip Codes', fontweight='bold')
axes[1].set_xlabel('Number of Zip Codes')

plt.tight_layout()
plt.show()

In [ ]:
# 8.6 Returns — Reason distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

returns['return_reason'].value_counts().plot.bar(ax=axes[0], color='#c0392b', edgecolor='white')
axes[0].set_title('Return Reasons', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

reviews['rating'].value_counts().sort_index().plot.bar(ax=axes[1], color='#f39c12', edgecolor='white')
axes[1].set_title('Review Rating Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Rating')

plt.suptitle('Returns & Reviews', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 8.7 Promotions overview
print('📊 Promotions Summary')
print(f"  Total promotions: {len(promotions)}")
print(f"  Types: {promotions['promo_type'].value_counts().to_dict()}")
print(f"  Channels: {promotions['promo_channel'].value_counts().to_dict()}")
print(f"  Stackable: {promotions['stackable_flag'].value_counts().to_dict()}")
print(f"  Discount range: {promotions['discount_value'].min()} – {promotions['discount_value'].max()}")
print(f"  Categories targeted: {promotions['applicable_category'].unique()}")
display(promotions.head(10))

## 9 — Revenue & COGS Time Series

In [ ]:
# 9.1 Daily Revenue & COGS time series (full historical period)
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(sales_train['Date'], sales_train['Revenue'], lw=0.5, color='#2980b9', alpha=0.7)
axes[0].plot(sales_train['Date'], sales_train['Revenue'].rolling(30).mean(),
             lw=2, color='#e74c3c', label='30-day MA')
axes[0].set_title('Daily Revenue (2012–2022)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Revenue (VND)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(sales_train['Date'], sales_train['COGS'], lw=0.5, color='#e67e22', alpha=0.7)
axes[1].plot(sales_train['Date'], sales_train['COGS'].rolling(30).mean(),
             lw=2, color='#8e44ad', label='30-day MA')
axes[1].set_title('Daily COGS (2012–2022)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('COGS (VND)')
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 9.2 Monthly aggregated Revenue & Gross Profit trend
monthly = sales_train.copy()
monthly['year'] = monthly['Date'].dt.year
monthly['month'] = monthly['Date'].dt.month
monthly['YearMonth'] = monthly['Date'].dt.to_period('M')
monthly_agg = monthly.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    COGS=('COGS', 'sum')
).reset_index()
monthly_agg['Gross_Profit'] = monthly_agg['Revenue'] - monthly_agg['COGS']
monthly_agg['Gross_Margin'] = monthly_agg['Gross_Profit'] / monthly_agg['Revenue']
monthly_agg['YearMonth_str'] = monthly_agg['YearMonth'].astype(str)

fig, ax1 = plt.subplots(figsize=(16, 6))

x = range(len(monthly_agg))
ax1.bar(x, monthly_agg['Revenue'] / 1e9, color='#3498db', alpha=0.6, label='Revenue')
ax1.bar(x, monthly_agg['COGS'] / 1e9, color='#e74c3c', alpha=0.4, label='COGS')
ax1.set_ylabel('Amount (Billion VND)', fontsize=12)
ax1.set_title('Monthly Revenue vs COGS with Gross Margin Trend', fontweight='bold', fontsize=13)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(x, monthly_agg['Gross_Margin'] * 100, color='#2ecc71', lw=2, marker='.', markersize=3, label='Gross Margin %')
ax2.set_ylabel('Gross Margin (%)', fontsize=12, color='#2ecc71')
ax2.legend(loc='upper right')

# Show only yearly ticks
yearly_ticks = [i for i, ym in enumerate(monthly_agg['YearMonth_str']) if ym.endswith('-01')]
ax1.set_xticks(yearly_ticks)
ax1.set_xticklabels([monthly_agg['YearMonth_str'].iloc[i][:4] for i in yearly_ticks], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 9.3 Annual Revenue trend with YoY growth
annual = sales_train.copy()
annual['year'] = annual['Date'].dt.year
annual_agg = annual.groupby('year').agg(
    Revenue=('Revenue', 'sum'),
    COGS=('COGS', 'sum')
).reset_index()
annual_agg['YoY_Growth'] = annual_agg['Revenue'].pct_change() * 100

fig, ax1 = plt.subplots(figsize=(12, 5))

bars = ax1.bar(annual_agg['year'], annual_agg['Revenue'] / 1e9,
               color='#3498db', edgecolor='white', alpha=0.8)
ax1.set_ylabel('Revenue (Billion VND)', fontsize=12)
ax1.set_title('Annual Revenue & YoY Growth Rate', fontweight='bold', fontsize=13)

# Annotate bars
for bar, val in zip(bars, annual_agg['Revenue'] / 1e9):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{val:.1f}B', ha='center', va='bottom', fontsize=8)

ax2 = ax1.twinx()
ax2.plot(annual_agg['year'], annual_agg['YoY_Growth'], color='#e74c3c',
         marker='o', lw=2, markersize=6, label='YoY Growth %')
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('YoY Growth (%)', fontsize=12, color='#e74c3c')
ax2.legend(loc='lower right')

ax1.set_xticks(annual_agg['year'])
plt.tight_layout()
plt.show()

display(annual_agg)

In [ ]:
# 9.4 Seasonality patterns — Revenue by Month (all years overlaid)
seasonal = sales_train.copy()
seasonal['year'] = seasonal['Date'].dt.year
seasonal['month'] = seasonal['Date'].dt.month

monthly_by_year = seasonal.groupby(['year', 'month'])['Revenue'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for year in sorted(monthly_by_year['year'].unique()):
    data = monthly_by_year[monthly_by_year['year'] == year]
    alpha = 0.3 if year < 2020 else 0.8
    lw = 1 if year < 2020 else 2.5
    ax.plot(data['month'], data['Revenue'] / 1e9, marker='.', alpha=alpha, lw=lw, label=str(year))

ax.set_title('Monthly Revenue by Year (Seasonality Pattern)', fontweight='bold', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (Billion VND)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Year')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 9.5 Day-of-week Revenue pattern
dow = sales_train.copy()
dow['day_of_week'] = dow['Date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_agg = dow.groupby('day_of_week')['Revenue'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db' if d not in ['Saturday', 'Sunday'] else '#e74c3c' for d in dow_order]
ax.bar(dow_order, dow_agg / 1e6, color=colors, edgecolor='white')
ax.set_title('Average Daily Revenue by Day of Week', fontweight='bold', fontsize=13)
ax.set_ylabel('Avg Revenue (Million VND)')
ax.set_xlabel('Day of Week')
for i, (d, v) in enumerate(zip(dow_order, dow_agg / 1e6)):
    ax.text(i, v, f'{v:.1f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 9.6 Orders volume over time
orders_ts = orders.copy()
orders_ts['YearMonth'] = orders_ts['order_date'].dt.to_period('M')
orders_monthly = orders_ts.groupby('YearMonth').size().reset_index(name='order_count')
orders_monthly['YearMonth_str'] = orders_monthly['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(range(len(orders_monthly)), orders_monthly['order_count'],
                alpha=0.4, color='#3498db')
ax.plot(range(len(orders_monthly)), orders_monthly['order_count'],
        lw=1.5, color='#2c3e50')
ax.set_title('Monthly Order Volume Over Time', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Orders')

yearly_ticks = [i for i, ym in enumerate(orders_monthly['YearMonth_str']) if ym.endswith('-01')]
ax.set_xticks(yearly_ticks)
ax.set_xticklabels([orders_monthly['YearMonth_str'].iloc[i][:4] for i in yearly_ticks], rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10 — Descriptive Statistics for Key Numeric Columns

In [ ]:
print('📊 Key Numeric Statistics')
print('\n--- Products ---')
display(products[['price', 'cogs', 'gross_margin']].describe())

print('\n--- Order Items ---')
display(order_items[['quantity', 'unit_price', 'discount_amount']].describe())

print('\n--- Payments ---')
display(payments[['payment_value', 'installments']].describe())

print('\n--- Returns ---')
display(returns[['return_quantity', 'refund_amount']].describe())

print('\n--- Reviews ---')
display(reviews[['rating']].describe())

print('\n--- Sales Train ---')
display(sales_train[['Revenue', 'COGS']].describe())

print('\n--- Web Traffic ---')
display(web_traffic[['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']].describe())

## 11 — Web Traffic & Inventory Quick Look

In [ ]:
# Web Traffic over time by source
wt = web_traffic.copy()
wt['YearMonth'] = wt['date'].dt.to_period('M')
wt_monthly = wt.groupby(['YearMonth', 'traffic_source'])['sessions'].sum().reset_index()
wt_pivot = wt_monthly.pivot(index='YearMonth', columns='traffic_source', values='sessions').fillna(0)

fig, ax = plt.subplots(figsize=(16, 5))
wt_pivot.plot.area(ax=ax, alpha=0.6, stacked=True)
ax.set_title('Monthly Web Sessions by Traffic Source', fontweight='bold', fontsize=13)
ax.set_ylabel('Sessions')
ax.legend(title='Source', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Inventory health overview
inv = inventory.copy()
inv_summary = inv.groupby('category').agg(
    avg_fill_rate=('fill_rate', 'mean'),
    avg_sell_through=('sell_through_rate', 'mean'),
    stockout_pct=('stockout_flag', 'mean'),
    overstock_pct=('overstock_flag', 'mean'),
    total_records=('product_id', 'count')
).round(4)

display(inv_summary.style.format({
    'avg_fill_rate': '{:.2%}',
    'avg_sell_through': '{:.2%}',
    'stockout_pct': '{:.2%}',
    'overstock_pct': '{:.2%}',
}).set_caption('📦 Inventory Health by Category'))

## 12 — Summary & Next Steps

### Key Findings from Phase 1 EDA:

1. **Data Quality**: _(Fill after running — note missing values, orphan FKs, etc.)_
2. **Revenue Trends**: _(Fill after running — overall trend direction, notable year)_
3. **Seasonality**: _(Fill after running — peak months, weekend effects)_
4. **Product Mix**: _(Fill after running — dominant categories/segments)_
5. **Customer Base**: _(Fill after running — demographics skew)_

### Next Steps:
- **Phase 2**: Answer 10 MCQ questions → `02_mcq_answers.ipynb`
- **Phase 3**: Deep EDA with 4-level analysis → `03_eda_deep.ipynb`
- **Phase 4**: Revenue/COGS forecasting model → `04_forecasting.ipynb`